# Threat Context Engineering v0.4 — Detection & Validation Workbench

**Colab profile:** GLiNER ON · Qwen ON · Threat Context Matrix · Telemetry Health · Detection Specification · Validation as Code · Drift.

The notebook keeps AI outputs review-only and uses safe defensive event fixtures for executable validation.

In [ ]:
import importlib, os, pathlib, site, subprocess, sys
os.environ.setdefault('TCE_AI_GLINER', '1')
os.environ.setdefault('TCE_AI_QWEN', '1')
os.environ.setdefault('TCE_QWEN_MODEL', 'auto')
os.environ.setdefault('TCE_QWEN_4BIT', '1')
REPO = 'https://github.com/Ridd1kulusC0d3r/Cyber-Threat-Context-Modeling.git'
BRANCH = os.environ.get('TCE_BRANCH', 'main')
ROOT = pathlib.Path('/content/Cyber-Threat-Context-Modeling')
if ROOT.exists() and (ROOT / '.git').exists():
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH,'--depth','1'],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','-B',BRANCH,'FETCH_HEAD'],check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','-e','.[colab]'],check=True)
SRC = ROOT / 'src'
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
site.addsitedir(str(SRC)); importlib.invalidate_caches()
import tce
print('TCE', tce.__version__, 'from', tce.__file__)
print('GLiNER:', os.environ['TCE_AI_GLINER'], '| Qwen:', os.environ['TCE_AI_QWEN'])
subprocess.run(['nvidia-smi'],check=False)


In [ ]:
from tce.io import load_case
from tce.validation import validate_entities
CASE_DIR = ROOT / 'examples/cases/enterprise-identity'
entities, errors = load_case(CASE_DIR)
errors += validate_entities(entities)
assert not errors, errors
print('Case entities:', len(entities), '| semantic validation: OK')


In [ ]:
from tce.conformance import evaluate_conformance
evaluate_conformance(entities, level='validated')


In [ ]:
from tce.telemetry import evaluate_case_telemetry
from tce.detection import compile_backlog
telemetry_health = evaluate_case_telemetry(entities)
detection_backlog = compile_backlog(entities)
display(telemetry_health)
display(detection_backlog)


In [ ]:
from tce.validation_harness import run_validation
validation_result = run_validation(entities, 'VAL-001', CASE_DIR)
validation_result


In [ ]:
from tce.detection import write_sigma
sigma_path = write_sigma(entities['DSP-001']['data'], '/content/DSP-001.yml')
print(sigma_path.read_text())


In [ ]:
from tce.history import build_snapshot, write_snapshot
snapshot = build_snapshot(entities)
write_snapshot(snapshot, '/content/tce-v0.4-snapshot.json')
snapshot


## AI review layer

GLiNER extracts entity candidates and Qwen proposes analytical candidates. Neither can silently modify source evidence.

In [ ]:
from tce.ai.config import AIConfig
from tce.ai.copilot import TCECopilot
config = AIConfig()
print(config)
copilot = TCECopilot(config)
print('AI configured. Models load lazily when analysis is requested.')


## Web Workbench — port 3000

The same case is exposed through the local FastAPI workbench with Context, Engineering, ATT&CK, AI and Graph views.

In [ ]:
import time, urllib.request
from IPython.display import HTML, display
PORT = 3000
os.environ['TCE_CASE_DIR'] = str(CASE_DIR.resolve())
if 'tce_ui_proc' in globals() and tce_ui_proc.poll() is None:
    tce_ui_proc.terminate()
    try: tce_ui_proc.wait(timeout=3)
    except subprocess.TimeoutExpired: tce_ui_proc.kill()
tce_ui_proc = subprocess.Popen([sys.executable,'-m','uvicorn','tce.webapp:app','--host','0.0.0.0','--port',str(PORT),'--log-level','warning'],cwd=str(ROOT),env=os.environ.copy())
for _ in range(30):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health',timeout=1) as response:
            if response.status == 200: break
    except Exception: time.sleep(1)
else: raise RuntimeError('TCE UI did not become ready')
from google.colab import output
proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
display(HTML(f'<a href="{proxy_url}" target="_blank" style="display:inline-block;padding:12px 18px;background:#23d1b5;color:#03100e;border-radius:10px;font-weight:700;text-decoration:none">Open TCE v0.4 Workbench</a>'))
print(proxy_url)
